# 📸 snap.py — Web Snapshot Tool (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmatek11/webscraper/blob/main/snap_colab_optimized.ipynb)

Backup stron (HTML + assets + screenshot) albo same screenshoty całych stron, np. **przed i po** zmianach u klienta.

**Jak używać:**
1. Uruchom **1. Instalacja** (raz na sesję, ~1-2 min).
2. Uruchom **2. Pobierz snap.py**, który ściąga aktualną wersję skryptu z GitHuba, więc poprawki w repo działają tu od razu.
3. W **3. Ustawienia i start** wklej adresy, wybierz tryb i uruchom.
4. **4. Wyniki** pobiera ZIP-y na komputer i/lub kopiuje je na Google Drive.

## 1. Instalacja

In [ ]:
%%capture install_log
# Playwright + Chromium z zależnościami systemowymi
!pip install -q playwright requests
!playwright install --with-deps chromium

# Fonty jak na zwykłym komputerze z Windows (Arial, Times, Verdana...) + emoji.
# Bez tego strony bez web-fontów renderują się innym krojem niż u klienta.
!echo "ttf-mscorefonts-installer msttcorefonts/accept-mscorefonts-eula select true" | debconf-set-selections
!apt-get install -y -qq fonts-liberation fonts-noto-color-emoji ttf-mscorefonts-installer > /dev/null 2>&1 || true
!fc-cache -f > /dev/null

In [ ]:
print("✅ Instalacja zakończona. Szczegóły: print(install_log.stdout)")

## 2. Pobierz snap.py

In [ ]:
BRANCH = "main"  #@param {type:"string"}

import urllib.request
url = f"https://raw.githubusercontent.com/mrmatek11/webscraper/{BRANCH}/snap.py"
urllib.request.urlretrieve(url, "snap.py")
print(f"✅ snap.py pobrany z gałęzi '{BRANCH}'")

## 3. Ustawienia i start

- **URL_LIST**: jeden adres na linię, linie z `#` są pomijane.
- **MODE**: `screenshots` to same PNG (najszybciej), `full` to HTML + assets + screenshot, `crawl` sam znajduje podstrony (sitemap + linki). Prefix `clean-` oznacza agresywne zamykanie popupów.
- **LABEL**: np. `przed` / `po`. Wyniki lądują w osobnych folderach, łatwo porównać.
- **VIEWPORT / SCALE**: `1920x1080` + `1` = typowy monitor Full HD. `1.25` = laptop z Windows (skalowanie 125%). `2` = ostry screen (Retina, plik ~4× większy).

In [ ]:
URL_LIST = """
https://biuro-onesta.pl
"""

MODE = "crawl"  #@param ["screenshots", "full", "crawl", "clean-screenshots", "clean-full", "clean-crawl"]
LABEL = "przed"  #@param {type:"string"}
VIEWPORT = "1920x1080"  #@param ["1920x1080", "1536x864", "1440x900", "1366x768"]
SCALE = 1  #@param [1, 1.25, 1.5, 2] {type:"raw"}
WORKERS = 3  #@param {type:"slider", min:1, max:6, step:1}
MAX_PAGES = 300  #@param {type:"integer"}
MAX_SCREENSHOT_HEIGHT = 30000  #@param {type:"integer"}

import importlib, re
from datetime import datetime
from pathlib import Path
import snap
importlib.reload(snap)

snap.load_config()
vw, vh = map(int, VIEWPORT.split('x'))
snap._CFG.update(
    workers=WORKERS, max_pages=MAX_PAGES, max_screenshot_height=MAX_SCREENSHOT_HEIGHT,
    viewport_width=vw, viewport_height=vh, device_scale_factor=float(SCALE),
)

urls = [l.strip() for l in URL_LIST.splitlines() if l.strip() and not l.strip().startswith('#')]
if MODE.replace('clean-', '', 1) == 'crawl':
    urls = snap.discover_urls(urls, MAX_PAGES)
urls = list(dict.fromkeys(urls))

label = re.sub(r'[^\w\-]', '_', LABEL.strip()) or 'wyniki'
OUTPUT_DIR = Path('/content/results') / f"{datetime.now():%Y-%m-%d}_{label}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"log: {snap.setup_logging(OUTPUT_DIR)}")
print(f"mode: {MODE} | pages: {len(urls)} | viewport: {vw}x{vh} @{SCALE}x | output: {OUTPUT_DIR}")

snap.run(urls, OUTPUT_DIR, mode=MODE)

## 4. Wyniki

In [ ]:
DOWNLOAD_ZIPS = True  #@param {type:"boolean"}
COPY_TO_DRIVE = False  #@param {type:"boolean"}
DRIVE_FOLDER = "Web_Backups"  #@param {type:"string"}

import shutil
zips = sorted(OUTPUT_DIR.glob('*.zip'))
for z in zips:
    print(f" - {z.name} ({z.stat().st_size/1024/1024:.1f} MB)")

if COPY_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    dest = Path('/content/drive/MyDrive') / DRIVE_FOLDER / OUTPUT_DIR.name
    dest.mkdir(parents=True, exist_ok=True)
    for f in zips + sorted(OUTPUT_DIR.glob('*.log')):
        shutil.copy2(f, dest)
    print(f"✅ skopiowano na Drive: {dest}")

if DOWNLOAD_ZIPS:
    from google.colab import files
    for z in zips:
        files.download(str(z))